<a href="https://colab.research.google.com/github/Ayushi1303/IN126001402_Data_Science-_Internship-February--2026/blob/main/Task_2_Sentiment_Analysis_using_NLP_Pipeline_%26_ML_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [21]:
df = pd.read_json("/archive.zip", lines=True)

print(df.head())
print(df.columns)

       reviewerID        asin      reviewerName helpful  \
0  A30TL5EWN6DFXT  120401325X         christina  [0, 0]   
1   ASY55RVNIL0UD  120401325X          emily l.  [0, 0]   
2  A2TMXE2AFO7ONB  120401325X             Erica  [0, 0]   
3   AWJ0WZQYMYFQ4  120401325X                JM  [4, 4]   
4   ATX7CZYFXI1KW  120401325X  patrice m rogoza  [2, 3]   

                                          reviewText  overall  \
0  They look good and stick good! I just don't li...        4   
1  These stickers work like the review says they ...        5   
2  These are awesome and make my phone look so st...        5   
3  Item arrived in great time and was in perfect ...        4   
4  awesome! stays on, and looks great. can be use...        5   

                                     summary  unixReviewTime   reviewTime  
0                                 Looks Good      1400630400  05 21, 2014  
1                      Really great product.      1389657600  01 14, 2014  
2                         

In [22]:
print("Shape:", df.shape)

# Check missing values
print(df.isnull().sum())

# Sample text
print(df['reviewText'][0])


Shape: (194439, 9)
reviewerID           0
asin                 0
reviewerName      3519
helpful              0
reviewText           0
overall              0
summary              0
unixReviewTime       0
reviewTime           0
dtype: int64
They look good and stick good! I just don't like the rounded shape because I was always bumping it and Siri kept popping up and it was irritating. I just won't buy a product like this again


# New section

In [24]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    words = text.split()

    # Remove stopwords + Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

# Apply preprocessing
df['clean_text'] = df['reviewText'].apply(clean_text)

print(df[['reviewText', 'clean_text']].head())

                                          reviewText  \
0  They look good and stick good! I just don't li...   
1  These stickers work like the review says they ...   
2  These are awesome and make my phone look so st...   
3  Item arrived in great time and was in perfect ...   
4  awesome! stays on, and looks great. can be use...   

                                          clean_text  
0  look good stick good dont like rounded shape a...  
1  sticker work like review say stick great stay ...  
2  awesome make phone look stylish used one far a...  
3  item arrived great time perfect condition howe...  
4  awesome stay look great used multiple apple pr...  


In [25]:
df['sentiment'] = df['overall'].apply(lambda x: 1 if x >= 4 else 0)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'], test_size=0.2, random_state=42
)


In [27]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [28]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [29]:
def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n{name} Results:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [30]:
lr = LogisticRegression()

evaluate_model(lr, X_train_bow, X_test_bow, y_train, y_test, "Logistic Regression (BoW)")
evaluate_model(lr, X_train_tfidf, X_test_tfidf, y_train, y_test, "Logistic Regression (TF-IDF)")



Logistic Regression (BoW) Results:
Accuracy: 0.8504422958239045
Precision: 0.873795490691365
Recall: 0.9406156634999496
F1 Score: 0.9059751681324366


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


NameError: name 'classification_report' is not defined

In [31]:
nb = MultinomialNB()

evaluate_model(nb, X_train_bow, X_test_bow, y_train, y_test, "Naive Bayes (BoW)")
evaluate_model(nb, X_train_tfidf, X_test_tfidf, y_train, y_test, "Naive Bayes (TF-IDF)")



Naive Bayes (BoW) Results:
Accuracy: 0.8193272989096894
Precision: 0.8899242865463017
Recall: 0.8719997314444929
F1 Score: 0.8808708331920377


NameError: name 'classification_report' is not defined

In [32]:
results = []

def store_results(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

store_results(LogisticRegression(), X_train_tfidf, X_test_tfidf, y_train, y_test, "LR TF-IDF")
store_results(MultinomialNB(), X_train_tfidf, X_test_tfidf, y_train, y_test, "NB TF-IDF")


results_df = pd.DataFrame(results)
print(results_df)

       Model  Accuracy  Precision    Recall  F1 Score
0  LR TF-IDF  0.859597   0.879749  0.946020  0.911682
1  NB TF-IDF  0.813979   0.809711  0.989761  0.890728
